In [1]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON, INPUT_CSV_PATHS, TARGET_RANGES
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np
import mlflow
import joblib
import tempfile
import os

from build_utils import *

In [2]:
key_columns=['date', 'home', 'away']

In [3]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"
encoder = TeamEncoder.load(team_encoder_path)

In [4]:
seasons=sorted(ALL_SEASONS)

In [5]:
test_dict=dict((k, pd.read_csv(f'{REPO_PATH}/{INPUT_CSV_PATHS[k]}')) for k in COMPETITIONS)

In [6]:
all_features_dict={}
for competition in COMPETITIONS:
    if competition not in test_dict:
        continue
    test_df = test_dict[competition]
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)
    
    scaler = joblib.load(os.path.join(f"{MODELS_PATH}/data_processors/{competition}", 'standard_scaler.pkl'))
    all_features[scaler.feature_names_in_] = scaler.transform(all_features[scaler.feature_names_in_])

    all_features_dict[competition] = all_features.copy().fillna(-1)

In [7]:
train_features=pd.read_csv(f"{REPO_PATH}/data/features/premier_league/all_combined_features_2017-24.csv")
train_features['date'] = pd.to_datetime(train_features['date'])
train_features = train_features.merge(all_features_dict['premier_league'][['home', 'away', 'date']], on=['home', 'away', 'date'], how='right').fillna(-1)

for competition, df in all_features_dict.items():
    all_features_dict[competition] = df[train_features.columns]

In [8]:
# Compare train_features and all_features_dict['premier_league'] (excluding home, away, date)
tf_values = train_features.drop(columns=['home', 'away', 'date']).values
af_values = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).values

# Find where they differ
diff_mask = 1-np.isclose(tf_values, af_values)
diff_indices = np.argwhere(diff_mask)
print(f"Number of differing values: {diff_indices.shape[0]}")
print("First 10 differences:")
train_feature_cols = train_features.drop(columns=['home', 'away', 'date']).columns.tolist()
all_feature_cols = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).columns.tolist()
for idx, (row, col) in enumerate(diff_indices[:10]):
    home = train_features.iloc[row]['home']
    away = train_features.iloc[row]['away']
    date = train_features.iloc[row]['date']
    tf_val = tf_values[row, col]
    af_val = af_values[row, col]
    col_name = train_feature_cols[col]
    col_name_af = all_feature_cols[col]

    print(f"Row {row}, Column '{col_name}, {col_name_af}': home={home}, away={away}, date={date}, train_features={tf_val}, all_features_dict={af_val}")

Number of differing values: 0
First 10 differences:


In [9]:
model_dict={
    'premier_league': {
        'away_goals':{
            'run_id': '24844ed16020415b883a9703df45777f',
            'artifact_path': 'model',
        },
        'home_goals':{
            'run_id': '326f4d632a884456974600b975b1b068',
            'artifact_path': 'model',
        },
    }
}

In [10]:
def load_joblib_model_from_mlflow(run_id, artifact_path, tracking_uri=None):
    """
    Fetch and load a joblib-dumped model from MLflow given experiment_id, run_id, and artifact_path.
    Optionally specify the MLflow tracking URI.
    Returns the loaded model.
    """
    if tracking_uri is not None:
        mlflow.set_tracking_uri(tracking_uri)
    client = mlflow.tracking.MlflowClient()
    # Download artifact to a temporary directory
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_path = client.download_artifacts(run_id, artifact_path, tmp_dir)
        # Find the first .joblib file in the artifact directory
        for root, _, files in os.walk(local_path):
            for file in files:
                if file.endswith('.joblib'):
                    model_path = os.path.join(root, file)
                    return joblib.load(model_path)
        raise FileNotFoundError("No .joblib model file found in the artifact path.")


In [11]:
predictions={}

In [13]:
for competition in COMPETITIONS:
    predictions[competition] = {}
    for target_name in TARGET_RANGES:
        model = load_joblib_model_from_mlflow(model_dict[competition][target_name]['run_id'],
                                                model_dict[competition][target_name]['artifact_path'],
                                                f'{REPO_PATH}/mlflow')
        prediction = model.predict_proba(all_features_dict[competition].drop(columns=key_columns))
        prediction = pd.DataFrame(prediction, columns=list(range(TARGET_RANGES[target_name][0], TARGET_RANGES[target_name][1]+1))+['other'])
        predictions[competition][target_name]=prediction

/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# Add home, away, date columns from test_dict to each predictions DataFrame
for competition in predictions:
    test_rows = all_features_dict[competition][['home', 'away', 'date']].reset_index(drop=True)
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        # Prepend home, away, date columns
        df = pd.concat([test_rows, df.reset_index(drop=True)], axis=1)
        predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [15]:
predictions['premier_league']['home_goals']

,home,away,date,0,1,2,3,4,5,other
0,Bournemouth,Leicester City,2025-05-25,0.149063,0.258561,0.298253,0.161182,0.117350,0.011969,0.003621
1,Fulham,Manchester City,2025-05-25,0.232827,0.200413,0.373395,0.095163,0.077701,0.014744,0.005757
2,Ipswich Town,West Ham,2025-05-25,0.263258,0.434291,0.188680,0.060357,0.035002,0.014841,0.003571
3,Liverpool,Crystal Palace,2025-05-25,0.134375,0.260804,0.259084,0.240313,0.067642,0.022642,0.015140
4,Manchester Utd,Aston Villa,2025-05-25,0.326785,0.299986,0.211111,0.084663,0.061738,0.011181,0.004537
5,Newcastle Utd,Everton,2025-05-25,0.128464,0.431857,0.159000,0.138324,0.120276,0.015327,0.006752
6,Nott'ham Forest,Chelsea,2025-05-25,0.159571,0.516036,0.196706,0.102014,0.016786,0.005733,0.003154
7,Southampton,Arsenal,2025-05-25,0.418347,0.174215,0.163400,0.184827,0.037830,0.017653,0.003728
8,Tottenham,Brighton,2025-05-25,0.297356,0.285364,0.237212,0.129521,0.035053,0.008641,0.006853
9,Wolves,Brentford,2025-05-25,0.196106,0.281202,0.258649,0.199145,0.036729,0.023387,0.004782


In [16]:
predictions['premier_league']['away_goals']

,home,away,date,0,1,2,3,4,5,6,other
0,Bournemouth,Leicester City,2025-05-25,0.315114,0.472188,0.121887,0.062773,0.016133,0.004655,0.003672,0.003578
1,Fulham,Manchester City,2025-05-25,0.106866,0.397433,0.268725,0.105850,0.083986,0.030065,0.004479,0.002597
2,Ipswich Town,West Ham,2025-05-25,0.157951,0.442277,0.276818,0.074209,0.023820,0.020314,0.002607,0.002005
3,Liverpool,Crystal Palace,2025-05-25,0.233276,0.553914,0.153224,0.036033,0.015546,0.004540,0.001881,0.001586
4,Manchester Utd,Aston Villa,2025-05-25,0.216772,0.213717,0.266520,0.196031,0.095239,0.005429,0.004334,0.001957
5,Newcastle Utd,Everton,2025-05-25,0.351440,0.279821,0.281033,0.067227,0.011578,0.004725,0.002039,0.002138
6,Nott'ham Forest,Chelsea,2025-05-25,0.211899,0.455137,0.205172,0.085633,0.022852,0.006041,0.008170,0.005096
7,Southampton,Arsenal,2025-05-25,0.245107,0.327364,0.201375,0.133906,0.041722,0.045938,0.002361,0.002229
8,Tottenham,Brighton,2025-05-25,0.198665,0.268450,0.428511,0.056852,0.035401,0.003814,0.006484,0.001822
9,Wolves,Brentford,2025-05-25,0.315997,0.197269,0.372964,0.075712,0.025861,0.007010,0.003550,0.001638


In [17]:
# Rename columns and update values in predictions DataFrames for each competition and target
for competition in predictions:
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        cols = df.columns.tolist()
        # Only rename and update non-metadata columns (assume first three are home, away, date)
        meta_cols = ['home', 'away', 'date']
        feature_cols = cols[3:]
        # Rename columns: first becomes lte_{original}, others become gt_{previous}
        new_cols = meta_cols.copy()
        if feature_cols:
            new_cols.append(f"lte_{feature_cols[0]}")
            for i in range(1, len(feature_cols)):
                new_cols.append(f"gt_{feature_cols[i-1]}")
        # Update values: first feature column stays, others become sum of itself and all to the right
        arr = df[feature_cols].values.copy() if feature_cols else None
        if arr is not None and arr.shape[1] > 0:
            for i in range(1, arr.shape[1]):
                arr[:, i] = arr[:, i:].sum(axis=1)
            df = pd.concat([df[meta_cols].reset_index(drop=True), pd.DataFrame(arr, columns=new_cols[3:])], axis=1)
            df.columns = new_cols
            predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [18]:
predictions['premier_league']['home_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5
0,Bournemouth,Leicester City,2025-05-25,0.149063,0.850937,0.592376,0.294123,0.132940,0.015590,0.003621
1,Fulham,Manchester City,2025-05-25,0.232827,0.767173,0.566760,0.193366,0.098202,0.020501,0.005757
2,Ipswich Town,West Ham,2025-05-25,0.263258,0.736742,0.302451,0.113771,0.053414,0.018412,0.003571
3,Liverpool,Crystal Palace,2025-05-25,0.134375,0.865625,0.604821,0.345737,0.105424,0.037783,0.015140
4,Manchester Utd,Aston Villa,2025-05-25,0.326785,0.673215,0.373230,0.162119,0.077456,0.015718,0.004537
5,Newcastle Utd,Everton,2025-05-25,0.128464,0.871536,0.439679,0.280679,0.142355,0.022079,0.006752
6,Nott'ham Forest,Chelsea,2025-05-25,0.159571,0.840430,0.324393,0.127687,0.025673,0.008887,0.003154
7,Southampton,Arsenal,2025-05-25,0.418347,0.581653,0.407438,0.244039,0.059212,0.021382,0.003728
8,Tottenham,Brighton,2025-05-25,0.297356,0.702644,0.417280,0.180069,0.050547,0.015494,0.006853
9,Wolves,Brentford,2025-05-25,0.196106,0.803894,0.522693,0.264044,0.064899,0.028170,0.004782


In [19]:
predictions['premier_league']['away_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5,gt_6
0,Bournemouth,Leicester City,2025-05-25,0.315114,0.684886,0.212698,0.090811,0.028037,0.011904,0.007250,0.003578
1,Fulham,Manchester City,2025-05-25,0.106866,0.893134,0.495702,0.226977,0.121127,0.037140,0.007076,0.002597
2,Ipswich Town,West Ham,2025-05-25,0.157951,0.842049,0.399772,0.122954,0.048745,0.024925,0.004611,0.002005
3,Liverpool,Crystal Palace,2025-05-25,0.233276,0.766724,0.212810,0.059586,0.023553,0.008007,0.003467,0.001586
4,Manchester Utd,Aston Villa,2025-05-25,0.216772,0.783228,0.569511,0.302991,0.106959,0.011720,0.006291,0.001957
5,Newcastle Utd,Everton,2025-05-25,0.351440,0.648560,0.368739,0.087707,0.020480,0.008902,0.004177,0.002138
6,Nott'ham Forest,Chelsea,2025-05-25,0.211899,0.788101,0.332964,0.127792,0.042159,0.019307,0.013266,0.005096
7,Southampton,Arsenal,2025-05-25,0.245107,0.754893,0.427530,0.226155,0.092249,0.050528,0.004589,0.002229
8,Tottenham,Brighton,2025-05-25,0.198665,0.801335,0.532884,0.104374,0.047522,0.012120,0.008306,0.001822
9,Wolves,Brentford,2025-05-25,0.315997,0.684003,0.486734,0.113770,0.038058,0.012197,0.005188,0.001638
